In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/pranaypoddar/amex-docs/test_data.parquet
/kaggle/input/datasets/pranaypoddar/amex-docs/add_event.parquet
/kaggle/input/datasets/pranaypoddar/amex-docs/data_dictionary.csv
/kaggle/input/datasets/pranaypoddar/amex-docs/offer_metadata.parquet
/kaggle/input/datasets/pranaypoddar/amex-docs/add_trans.parquet
/kaggle/input/datasets/pranaypoddar/amex-docs/train_data.parquet


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import gc
import logging
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Configuration ---
warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('robust_pipeline.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [4]:
DATA_PATH = "/kaggle/input/datasets/pranaypoddar/amex-docs"
logger.info("Starting data loading and preparation.")
try:
    train = pd.read_parquet(f"{DATA_PATH}/train_data.parquet")
    logger.info(f"Loaded train_data.parquet with shape: {train.shape}")
    test = pd.read_parquet(f"{DATA_PATH}/test_data.parquet")
    logger.info(f"Loaded test_data.parquet with shape: {test.shape}")
    events = pd.read_parquet(f"{DATA_PATH}/add_event.parquet")
    logger.info(f"Loaded add_event.parquet with shape: {events.shape}")
    trans = pd.read_parquet(f"{DATA_PATH}/add_trans.parquet")
    logger.info(f"Loaded add_trans.parquet with shape: {trans.shape}")
    offers = pd.read_parquet(f"{DATA_PATH}/offer_metadata.parquet")
    logger.info(f"Loaded offer_metadata.parquet with shape: {offers.shape}")
except Exception as e:
    logger.error(f"Error loading data: {str(e)}")
    raise

2026-10-12 10:38:34,723 - INFO - Starting data loading and preparation.
2026-10-12 10:38:47,064 - INFO - Loaded train_data.parquet with shape: (770164, 372)
2026-10-12 10:38:53,274 - INFO - Loaded test_data.parquet with shape: (369301, 371)
2026-10-12 10:39:10,056 - INFO - Loaded add_event.parquet with shape: (21457473, 5)
2026-10-12 10:39:12,660 - INFO - Loaded add_trans.parquet with shape: (6339465, 9)
2026-10-12 10:39:12,678 - INFO - Loaded offer_metadata.parquet with shape: (4164, 12)


In [5]:
# Standardize IDs to string for safe merging
for df_name, df in zip(['train', 'test', 'events', 'trans', 'offers'], [train, test, events, trans, offers]):
    for col in ['id2', 'id3']:
        if col in df.columns:
            df[col] = df[col].astype(str)
logger.info("Standardized 'id2' and 'id3' columns to string type.")

# Prepare target variable
train['y'] = pd.to_numeric(train['y'], errors='coerce').fillna(0).astype(int)

# Convert timestamps
train['id4'] = pd.to_datetime(train['id4'], errors='coerce')
test['id4'] = pd.to_datetime(test['id4'], errors='coerce')
trans['f370'] = pd.to_datetime(trans['f370'], errors='coerce')
offers['id12'] = pd.to_datetime(offers['id12'], errors='coerce') # start date
offers['id13'] = pd.to_datetime(offers['id13'], errors='coerce') # end date
logger.info("Converted timestamp columns to datetime objects.")

logger.info(f"Data preparation complete. Final shapes - Train: {train.shape}, Test: {test.shape}")

2026-10-12 10:43:18,625 - INFO - Standardized 'id2' and 'id3' columns to string type.
2026-10-12 10:43:19,905 - INFO - Converted timestamp columns to datetime objects.
2026-10-12 10:43:19,906 - INFO - Data preparation complete. Final shapes - Train: (770164, 372), Test: (369301, 371)


In [6]:
# --- 2. Feature Engineering ---
def create_features(df, offers, trans, events):
    """Creates a rich set of features for the ranking model."""
    logger.info(f"Starting feature engineering on dataframe with shape {df.shape}.")
    
    # Merge offer metadata first
    df = df.merge(offers, on='id3', how='left')
    logger.info(f"Shape after merging offers: {df.shape}")

    # --- A. Basic Temporal Features ---
    df['dayofweek'] = df['id4'].dt.dayofweek
    df['hour'] = df['id4'].dt.hour
    logger.info("Created basic temporal features: dayofweek, hour.")
    
    # --- B. Industry Code (id8) Alignment Features ---
    logger.info("Creating id8 (Industry) alignment features.")
    customer_industry_spend = trans.groupby(['id2', 'id8'])['f367'].sum().reset_index()
    customer_total_spend = trans.groupby('id2')['f367'].sum().reset_index().rename(columns={'f367': 'total_spend'})
    customer_industry_spend = customer_industry_spend.merge(customer_total_spend, on='id2', how='left')
    customer_industry_spend['industry_spend_proportion'] = customer_industry_spend['f367'] / customer_industry_spend['total_spend']
    
    df = df.merge(customer_industry_spend[['id2', 'id8', 'industry_spend_proportion']], on=['id2', 'id8'], how='left')
    df['industry_match'] = df['industry_spend_proportion'].notna().astype(int)
    
    # --- C. Behavioral, Fatigue & Session Features ---
    logger.info("Creating behavioral and fatigue features.")
    df = df.sort_values(by=['id2', 'id4']).reset_index(drop=True)
    df['offer_fatigue_user'] = df.groupby(['id2', 'id3']).cumcount()
    df['campaign_fatigue_user'] = df.groupby(['id2', 'id9']).cumcount()
    df['session_impression_gap_secs'] = df.groupby('id2')['id4'].diff().dt.total_seconds()
    
    # --- D. Cold-Start & Global Features ---
    logger.info("Creating cold-start and global features.")
    df['offer_age_days'] = (df['id4'] - df['id12']).dt.days
    df['offer_duration_days'] = (df['id13'] - df['id12']).dt.days
    
    # --- E. Transaction and Event Aggregations ---
    logger.info("Creating transaction and event aggregations.")
    agg_trans = trans.groupby('id2')['f367'].agg(['mean', 'std', 'sum', 'count']).reset_index()
    agg_trans.columns = ['id2', 'trans_mean', 'trans_std', 'trans_sum', 'trans_count']
    df = df.merge(agg_trans, on='id2', how='left')

    events['click_flag'] = events['id7'].notnull().astype(int)
    event_aggs = events.groupby('id2').agg(click_rate=('click_flag', 'mean'), click_count=('click_flag', 'sum'))
    df = df.merge(event_aggs, on='id2', how='left')

    # --- F. Interaction Features ---
    df['offer_popularity'] = df.groupby('id3')['id2'].transform('count')
    df['customer_offer_frequency'] = df.groupby('id2')['id3'].transform('count')

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    logger.info(f"Feature engineering complete. Final shape: {df.shape}")
    return df

In [7]:
train_fe = create_features(train, offers, trans, events)
test_fe = create_features(test, offers, trans, events)

2026-10-12 10:44:29,190 - INFO - Starting feature engineering on dataframe with shape (770164, 372).
2026-10-12 10:44:41,082 - INFO - Shape after merging offers: (770164, 383)
2026-10-12 10:44:41,130 - INFO - Created basic temporal features: dayofweek, hour.
2026-10-12 10:44:41,130 - INFO - Creating id8 (Industry) alignment features.
2026-10-12 10:45:00,200 - INFO - Creating behavioral and fatigue features.
2026-10-12 10:45:09,275 - INFO - Creating cold-start and global features.
2026-10-12 10:45:09,316 - INFO - Creating transaction and event aggregations.
2026-10-12 10:46:01,646 - INFO - Feature engineering complete. Final shape: (770164, 400)
2026-10-12 10:46:01,701 - INFO - Starting feature engineering on dataframe with shape (369301, 371).
2026-10-12 10:46:05,648 - INFO - Shape after merging offers: (369301, 382)
2026-10-12 10:46:05,671 - INFO - Created basic temporal features: dayofweek, hour.
2026-10-12 10:46:05,672 - INFO - Creating id8 (Industry) alignment features.
2026-10-12 

In [8]:
def apply_text_and_target_encoding(train_df, test_df, target_col='y', group_col='id2'):
    """Applies TF-IDF and GroupKFold Target Encoding."""
    logger.info("Applying TF-IDF and Target Encoding.")
    
    # TF-IDF on 'f378'
    tfidf = TfidfVectorizer(max_features=30, min_df=5, dtype=np.float32)
    tfidf_train = tfidf.fit_transform(train_df['f378'].fillna('missing'))
    tfidf_test = tfidf.transform(test_df['f378'].fillna('missing'))
    
    tfidf_train_df = pd.DataFrame(tfidf_train.toarray(), columns=[f'tfidf_{i}' for i in range(tfidf_train.shape[1])])
    tfidf_test_df = pd.DataFrame(tfidf_test.toarray(), columns=[f'tfidf_{i}' for i in range(tfidf_test.shape[1])])
    
    train_df = pd.concat([train_df.reset_index(drop=True), tfidf_train_df], axis=1)
    test_df = pd.concat([test_df.reset_index(drop=True), tfidf_test_df], axis=1)
    logger.info(f"Applied TF-IDF. Train shape: {train_df.shape}, Test shape: {test_df.shape}")
    
    # GroupKFold Target Encoding for 'id3'
    logger.info("Starting GroupKFold Target Encoding for 'id3'.")
    kf = GroupKFold(n_splits=5)
    global_mean = train_df[target_col].mean()
    
    train_df['id3_te'] = 0.0
    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df, train_df[target_col], train_df[group_col])):
        logger.info(f"Processing fold {fold+1}/5 for target encoding.")
        means = train_df.iloc[tr_idx].groupby('id3')[target_col].mean()
        train_df.loc[val_idx, 'id3_te'] = train_df.loc[val_idx, 'id3'].map(means).fillna(global_mean)
        
    full_train_means = train_df.groupby('id3')[target_col].mean()
    test_df['id3_te'] = test_df['id3'].map(full_train_means).fillna(global_mean)
    logger.info("Target encoding complete.")
    
    return train_df, test_df

In [9]:
train_processed, test_processed = apply_text_and_target_encoding(train_fe, test_fe)

2026-10-12 10:47:52,520 - INFO - Applying TF-IDF and Target Encoding.
2026-10-12 10:48:05,286 - INFO - Applied TF-IDF. Train shape: (770164, 430), Test shape: (369301, 429)
2026-10-12 10:48:05,287 - INFO - Starting GroupKFold Target Encoding for 'id3'.
2026-10-12 10:48:05,806 - INFO - Processing fold 1/5 for target encoding.
2026-10-12 10:48:09,457 - INFO - Processing fold 2/5 for target encoding.
2026-10-12 10:48:13,095 - INFO - Processing fold 3/5 for target encoding.
2026-10-12 10:48:16,730 - INFO - Processing fold 4/5 for target encoding.
2026-10-12 10:48:20,381 - INFO - Processing fold 5/5 for target encoding.
2026-10-12 10:48:24,059 - INFO - Target encoding complete.


In [10]:
gc.collect

<function gc.collect(generation=2)>

In [11]:
exclude_cols = ['id1', 'id2', 'id3', 'id4', 'id5', 'id9', 'id10', 'id11', 'id12', 'id13', 'f378', 'start', 'end', 'y']
features = [col for col in train_processed.columns if col not in exclude_cols]
features = [f for f in features if train_processed[f].dtype in ['int64', 'float64', 'int32', 'float32', 'int8', 'bool']]
logger.info(f"Training with {len(features)} features.")

2026-10-12 10:48:30,286 - INFO - Training with 51 features.


In [12]:
# --- 4. Model Training with Optuna ---
def train_lgbm_with_optuna(train_df, test_df, features, n_trials=10):
    """Trains a LightGBM Ranker with Optuna for hyperparameter tuning."""
    logger.info(f"Starting model training with Optuna for {n_trials} trials.")
    
    X = train_df[features].copy()
    y = train_df['y'].copy()
    X_test = test_df[features].copy()
    groups = train_df['id2'].copy()
    
    # NEW LOGGING: Log data shapes before training
    logger.info(f"Training data shape (X): {X.shape}")
    logger.info(f"Test data shape (X_test): {X_test.shape}")
    
    for col in X.columns:
        if X[col].isnull().any():
            median_val = X[col].median()
            X[col].fillna(median_val, inplace=True)
            X_test[col].fillna(median_val, inplace=True)
    logger.info("Filled NaN values using column medians.") # NEW LOGGING

    def objective(trial):
        params = {
            'objective': 'lambdarank', 'metric': 'map', 'boosting_type': 'gbdt',
            'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
            'num_leaves': trial.suggest_int('num_leaves', 31, 100),
            'max_depth': trial.suggest_int('max_depth', 7, 11),
            'min_child_samples': trial.suggest_int('min_child_samples', 50, 150),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-6, 5.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-6, 5.0, log=True),
            'random_state': 42, 'verbose': -1, 'bagging_freq': 1
        }
        
        gkf = GroupKFold(n_splits=4)
        oof_map_scores = []

        for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            group_tr = groups.iloc[train_idx].value_counts(sort=False).values
            group_val = groups.iloc[val_idx].value_counts(sort=False).values

            model = lgb.LGBMRanker(**params)
            model.fit(X_train, y_train, group=group_tr,
                      eval_set=[(X_val, y_val)], eval_group=[group_val],
                      eval_metric='map', eval_at=[7],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            
            map_at_7 = model.best_score_['valid_0']['map@7']
            oof_map_scores.append(map_at_7)
        
        mean_score = np.mean(oof_map_scores)
        logger.info(f"Optuna Trial {trial.number} finished with MAP@7: {mean_score}") # NEW LOGGING
        return mean_score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    logger.info(f"Best trial MAP@7: {study.best_value}")
    logger.info(f"Best params: {study.best_params}")
    
    logger.info("Training final model on all data with best parameters.")
    best_params = study.best_params
    best_params.update({'objective': 'lambdarank', 'metric': 'map'})
    
    final_model = lgb.LGBMRanker(**best_params)
    group_full = groups.value_counts(sort=False).values
    final_model.fit(X, y, group=group_full)
    logger.info("Final model training complete.") # NEW LOGGING
    
    test_preds = final_model.predict(X_test)
    
    return test_preds, final_model, study.best_params

In [13]:
test_predictions, model, _ = train_lgbm_with_optuna(train_processed, test_processed, features)

2026-10-12 10:48:41,590 - INFO - Starting model training with Optuna for 10 trials.
2026-10-12 10:48:41,779 - INFO - Training data shape (X): (770164, 51)
2026-10-12 10:48:41,780 - INFO - Test data shape (X_test): (369301, 51)
2026-10-12 10:48:41,927 - INFO - Filled NaN values using column medians.
[I 2026-10-12 10:48:41,928] A new study created in memory with name: no-name-b79f3821-96a1-43ec-8984-0ca6a8c73092
2026-10-12 10:49:34,143 - INFO - Optuna Trial 0 finished with MAP@7: 0.9522503032028263
[I 2026-10-12 10:49:34,146] Trial 0 finished with value: 0.9522503032028263 and parameters: {'n_estimators': 1934, 'learning_rate': 0.04961860767986279, 'num_leaves': 99, 'max_depth': 10, 'min_child_samples': 138, 'feature_fraction': 0.7077554939406187, 'bagging_fraction': 0.705845588335726, 'lambda_l1': 0.0026581381117312584, 'lambda_l2': 1.187988765563684e-06}. Best is trial 0 with value: 0.9522503032028263.
2026-10-12 10:50:44,984 - INFO - Optuna Trial 1 finished with MAP@7: 0.9525389860835

In [14]:
def create_submission(test_df, predictions, filename='submission_robust.csv'):
    """Creates the submission file in the required format."""
    logger.info(f"Creating submission file: {filename}")
    submission = test_df[['id1', 'id2', 'id3', 'id5']].copy()
    submission['pred'] = predictions
    
    # Group-wise normalization for ranking
    submission['pred'] = submission.groupby('id2')['pred'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
    )
    submission['id5'] = pd.to_datetime(submission['id5'], errors='coerce').dt.strftime('%m/%d/%y')
    
    submission.to_csv(filename, index=False)
    logger.info(f"Submission file saved to {filename}. Shape: {submission.shape}")
    return submission

In [15]:
_ = create_submission(test_processed, test_predictions)

importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
logger.info(f"Top 20 Feature Importances:\n{importance_df.head(20)}")

logger.info("====== Pipeline Finished Successfully ======")

2026-10-12 11:10:56,584 - INFO - Creating submission file: submission_robust.csv
2026-10-12 11:11:04,429 - INFO - Submission file saved to submission_robust.csv. Shape: (369301, 5)
2026-10-12 11:11:04,437 - INFO - Top 20 Feature Importances:
                        feature  importance
9   session_impression_gap_secs       15204
50                       id3_te       11821
4                          hour       10124
18             offer_popularity        8062
10               offer_age_days        7725
11          offer_duration_days        5052
3                     dayofweek        2681
8         campaign_fatigue_user        1658
1                          f376        1557
29                      tfidf_9        1309
48                     tfidf_28        1108
47                     tfidf_27         706
19     customer_offer_frequency         640
28                      tfidf_8         575
49                     tfidf_29         463
20                      tfidf_0         426
7         